# Build background transform artifacts

Ноутбук для генерации двух файлов, которые потом будут использоваться в `vdjdb transform`:

- `<chain>_background_transform.joblib`
- `<chain>_background_transform_bg_umap_<N>.npy`


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'redcea' / 'src'))

from vdjdb_redcea.compat import BackgroundTransform
from vdjdb_redcea.io import prepare_background_umap


In [ ]:
CHAINS = ['TRA', 'TRB']

BACKGROUND_EMBEDDING_DIR = Path('/projects/immunestatus/vdjdb_olga/tcremp')
OUTPUT_ROOT = REPO_ROOT / 'results' / 'redcea'

BACKGROUND_FILES = {
    'TRA': 'tra_background_100k_embeddings.parquet',
    'TRB': 'trb_background_100k_embeddings.parquet',
}

N_BG_POINTS = 100000
CLUSTER_PC_COMPONENTS = 50
RANDOM_STATE = 42


In [ ]:
def _numeric_embedding_columns(df: pd.DataFrame) -> list[str]:
    candidates = []
    for col in df.columns:
        lower = str(col).lower()
        if pd.api.types.is_numeric_dtype(df[col]) and any(token in lower for token in ('emb', 'dim', 'feat', 'pc', 'latent')):
            candidates.append(col)
    return candidates


def dataframe_to_embedding_matrix(df: pd.DataFrame) -> tuple[np.ndarray, str]:
    vector_columns = [
        col
        for col in df.columns
        if str(col).lower() in {'embedding', 'embeddings', 'vector', 'vectors', 'repr', 'representation'}
    ]

    for col in vector_columns:
        series = df[col].dropna()
        if series.empty:
            continue
        first_value = series.iloc[0]
        if isinstance(first_value, (list, tuple, np.ndarray)):
            matrix = np.vstack(df[col].map(np.asarray))
            return matrix.astype(np.float32), col

    numeric_cols = _numeric_embedding_columns(df)
    if numeric_cols:
        return df[numeric_cols].to_numpy(dtype=np.float32), f'numeric columns ({len(numeric_cols)})'

    numeric_df = df.select_dtypes(include=['number'])
    if numeric_df.shape[1] >= 8:
        return numeric_df.to_numpy(dtype=np.float32), 'all numeric columns'

    raise ValueError('Could not infer embedding columns: ' + ', '.join(map(str, df.columns)))


def build_background_artifacts(chain: str) -> dict[str, object]:
    chain = chain.upper()
    embedding_path = BACKGROUND_EMBEDDING_DIR / BACKGROUND_FILES[chain]
    if not embedding_path.exists():
        raise FileNotFoundError(f'Background embedding parquet not found: {embedding_path}')

    df = pd.read_parquet(embedding_path)
    bg_emb, embedding_source = dataframe_to_embedding_matrix(df)

    tcremp_dir = OUTPUT_ROOT / 'tcremp'
    tcremp_dir.mkdir(parents=True, exist_ok=True)
    transform_path = tcremp_dir / f'{chain.lower()}_background_transform.joblib'

    transform = BackgroundTransform(
        chain=chain,
        n_pca_components=CLUSTER_PC_COMPONENTS,
        random_state=RANDOM_STATE,
    )
    transform.fit(bg_emb)
    transform.save(transform_path)

    bg_pca = transform.background_pca_
    if bg_pca is None:
        bg_pca = transform.transform_pca(bg_emb)

    n_bg = min(len(bg_pca), N_BG_POINTS)
    bg_umap = prepare_background_umap(transform, bg_pca, n_bg, transform_path=transform_path)
    umap_cache_path = transform_path.with_name(f'{transform_path.stem}_bg_umap_{len(bg_umap)}.npy')

    return {
        'chain': chain,
        'embedding_path': embedding_path,
        'embedding_source': embedding_source,
        'n_rows': len(bg_emb),
        'n_dims': int(bg_emb.shape[1]),
        'transform_path': transform_path,
        'umap_cache_path': umap_cache_path,
        'umap_shape': tuple(bg_umap.shape),
    }


In [ ]:
results = [build_background_artifacts(chain) for chain in CHAINS]
pd.DataFrame(results)
